# Import

In [1]:
import pandas as pd

# Charger le fichier CSV avec le bon séparateur
df = pd.read_csv("../../create_data/dataset.csv", sep="\\|\\|\\|", engine="python")

# Ne conserver que les colonnes utiles
df = df[['modern', 'old_french']].dropna()

# Nettoyage basique (optionnel mais recommandé)
df['modern'] = df['modern'].str.strip()
df['modern'] = df['modern'].replace("traduction en ancien français :","")
df['old_french'] = df['old_french'].str.strip().replace("traduction en ancien français :","")

df.to_csv("cleaned_data.csv", sep="|", index=False)
# Vérification
print(df.sample(3))

                                                 modern  \
924   Je m'appelle Madeleine Dupuis et, derrière mes...   
3339  "Hey, c'est Alex, l'étudiant en médecine qui s...   
1274  "Hey, c'est Max. Écoute, je voulais te parler ...   

                                             old_french  
924   Je me nomme Madeleine Dupuis et, derrière mes ...  
3339  "Holla, c'est Alex, l'escolier en medicine qui...  
1274  Certes, c'est Max. Escoute, je voloie te parle...  


# Vieux truc avec cartes graphique pas opti

In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("t5-small", use_fast=False)

/home/utilisateur/Documents/Simplon/projet_nlp/EULA-vaaag-/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [3]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, Trainer, TrainingArguments
from torch.utils.data import Dataset
import torch

class TranslationDataset(Dataset):
    def __init__(self, modern_texts, old_french_texts, tokenizer, max_length=128):
        self.modern_texts = modern_texts
        self.old_french_texts = old_french_texts
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.modern_texts)
    
    def __getitem__(self, idx):
        modern = self.modern_texts[idx]
        old_french = self.old_french_texts[idx]
        modern_text = f"translate French to OldFrench: {modern}"
        
        # Tokenisation
        inputs = self.tokenizer(
            modern_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        targets = self.tokenizer(
            old_french, 
            max_length=self.max_length, 
            padding='max_length', 
            truncation=True, 
            return_tensors='pt'
        )
        
        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'labels': targets['input_ids'].flatten()
        }

# Modèle basé sur mT5 ou mBERT
model = AutoModelForSeq2SeqLM.from_pretrained('t5-small')

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.


OSError: [Errno 28] No space left on device: '/tmp/tmpg7eyqdag'

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
test_df, val_df = train_test_split(val_df, test_size=0.5, random_state=42)

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, Trainer, TrainingArguments, TrainerCallback

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=10,
    per_device_train_batch_size=1,      # Encore plus petit
    per_device_eval_batch_size=1,       # Encore plus petit
    gradient_accumulation_steps=8,      # Simule batch_size=8
    learning_rate=5e-5,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    eval_strategy='steps',              # Changé de 'epoch' à 'steps'
    eval_steps=100,                     # Évalue moins souvent
    save_strategy='steps',
    save_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    dataloader_pin_memory=False,        # Économise la mémoire
    remove_unused_columns=True,
)

In [ ]:
train_dataset = TranslationDataset(
    modern_texts=train_df["modern"].tolist(),
    old_french_texts=train_df["old_french"].tolist(),
    tokenizer=tokenizer,
    max_length=64
)

val_dataset = TranslationDataset(
    modern_texts=val_df["modern"].tolist(),
    old_french_texts=val_df["old_french"].tolist(),
    tokenizer=tokenizer,
    max_length=64
)


In [ ]:
from sacrebleu import corpus_bleu
from rouge_score import rouge_scorer
import nltk
import numpy as np
from typing import Dict
import torch

def evaluate_model(predictions, references):
    # BLEU Score
    bleu = corpus_bleu(predictions, [references])
    
    # ROUGE Score
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = [scorer.score(ref, pred) for ref, pred in zip(references, predictions)]
    
    # Similarité de caractères (important pour l'ancien français)
    char_similarity = [
        len(set(ref) & set(pred)) / len(set(ref) | set(pred)) if len(set(ref) | set(pred)) > 0 else 0
        for ref, pred in zip(references, predictions)
    ]
    
    return {
        'bleu': bleu.score,
        'rouge1': np.mean([s['rouge1'].fmeasure for s in rouge_scores]),
        'char_similarity': np.mean(char_similarity)
    }

def compute_metrics(eval_pred) -> Dict:
    predictions, labels = eval_pred
    
    # CORRECTION: Convertir les logits en tokens IDs
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    
    # Si predictions contient des logits, prendre l'argmax
    if len(predictions.shape) == 3:  # [batch, seq_len, vocab_size]
        predictions = np.argmax(predictions, axis=-1)
    
    # Remplacer les labels -100 par le token pad
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    
    # Décoder les prédictions et labels
    try:
        decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        
        # Nettoyer les textes vides
        decoded_preds = [pred.strip() if pred.strip() else "vide" for pred in decoded_preds]
        decoded_labels = [label.strip() if label.strip() else "vide" for label in decoded_labels]
        
        return evaluate_model(decoded_preds, decoded_labels)
    
    except Exception as e:
        print(f"Erreur dans compute_metrics: {e}")
        print(f"Shape predictions: {predictions.shape}")
        print(f"Shape labels: {labels.shape}")
        print(f"Type predictions: {type(predictions)}")
        
        # Retourner des métriques par défaut en cas d'erreur
        return {
            'bleu': 0.0,
            'rouge1': 0.0,
            'char_similarity': 0.0
        }

In [ ]:
print(torch.cuda.get_device_name(0)) 

Quadro T1000


In [ ]:
import gc

# Nettoie la mémoire GPU
torch.cuda.empty_cache()
gc.collect()

# Affiche l'utilisation mémoire
print(f"Mémoire GPU utilisée: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"Mémoire GPU réservée: {torch.cuda.memory_reserved()/1024**3:.2f} GB")

Mémoire GPU utilisée: 0.00 GB
Mémoire GPU réservée: 0.00 GB


In [ ]:
# import torch

# # Force l'utilisation du CPU
# device = torch.device('cpu')
# model = AutoModelForSeq2SeqLM.from_pretrained('t5-small')
# model.to(device)

# # Modifie les training_args
# training_args = TrainingArguments(
#     # ... autres paramètres
#     no_cuda=True,  # Force CPU
#     fp16=False,    # FP16 non supporté sur CPU
#     per_device_train_batch_size=4,  # Plus grand sur CPU
#     per_device_eval_batch_size=4,
# )

In [ ]:

class MemoryCleanupCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        torch.cuda.empty_cache()
        gc.collect()
        print(f"Mémoire nettoyée - Époque {state.epoch}")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

/tmp/ipykernel_142511/2535040372.py:7: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss,Validation Loss,Bleu,Rouge1,Char Similarity
100,No log,2.540024,32.996757,0.574139,0.816145
200,No log,2.216151,35.838143,0.607213,0.862271
300,No log,2.002463,36.736844,0.616754,0.867327
400,No log,1.836558,38.418433,0.629265,0.873750


SafetensorError: Error while serializing: IoError(Os { code: 28, kind: StorageFull, message: "No space left on device" })

In [ ]:
# Exemple d'entrée
modern_text = "Salut tout le monde ! Aujourd'hui on a fait un pique-nique dans le jardin. Après j'ai été malade, je me suis vidé par tous les trous..."
input_text = f"translate French to OldFrench: {modern_text}"

# Tokenisation
inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True)

# Génération (inférence)
with torch.no_grad():
    output_ids = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=128,
        num_beams=4,
        early_stopping=True
    )

# Décodage
output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("📝 Texte d'origine :", modern_text)
print("🏰 Traduction en vieux français :", output_text)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu! (when checking argument for argument index in method wrapper_CUDA__index_select)